In [3]:
import os
import psycopg2
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


file_path = "Telco_Customer_churn_Data.csv"   

telecom_df = pd.read_csv(file_path)

def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",  
    host="localhost",
    port="5432"
)
cur = conn.cursor()


table_name = "telecom_customers"
columns = telecom_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE "{table_name}" (
  {sql_columns}
);
"""

cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
cur.execute(create_stmt)
conn.commit()

print("Table recreated with schema:")
print(create_stmt)


columns_list = list(telecom_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
quoted_cols = ', '.join([f'"{col}"' for col in columns_list])

insert_stmt = f"""
INSERT INTO "{table_name}" ({quoted_cols})
VALUES ({placeholders})
"""

for _, row in telecom_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
print("Data inserted successfully")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)

cur.close()
conn.close()

print("Data fetched from DB:")
print(df_from_db.head())


Table recreated with schema:

CREATE TABLE "telecom_customers" (
  "customerID" TEXT,
  "gender" TEXT,
  "SeniorCitizen" INT,
  "Partner" TEXT,
  "Dependents" TEXT,
  "tenure" INT,
  "PhoneService" TEXT,
  "MultipleLines" TEXT,
  "InternetService" TEXT,
  "OnlineSecurity" TEXT,
  "OnlineBackup" TEXT,
  "DeviceProtection" TEXT,
  "TechSupport" TEXT,
  "StreamingTV" TEXT,
  "StreamingMovies" TEXT,
  "Contract" TEXT,
  "PaperlessBilling" TEXT,
  "PaymentMethod" TEXT,
  "MonthlyCharges" FLOAT,
  "TotalCharges" TEXT,
  "Churn" TEXT
);

Data inserted successfully
Data fetched from DB:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU 